# 1. Import Packages

In [11]:
import os
import biom
import qiime2 as q2
import pandas as pd

from qiime2 import Visualization

data_dir = 'updog_data' 
    
%matplotlib inline

In [ ]:
!wget -O "$data_dir/mags-table-filtered-milo-no-unassigned.qza" "https://polybox.ethz.ch/index.php/s/88arfy68xK7Ydz7/download"

In [ ]:
!wget -O "$data_dir/updog_metadata.tsv" "https://polybox.ethz.ch/index.php/s/pna5PZy62SfGcq5/download"

# 2. Training and evaluating classifiers

### 2.1 Training classifier to predict  `subsistence_mode` with microbial composition

We aim to use 80% of our samples as a train set to fit the classifier and the remaining 20% as a test set to evaluate its modelling performance.

We have 126 samples

The goal is to predict the metadata column `subsistence_mode` given the microbial composition. The microbial composition in our case is a FeatureTable[Frequency] artifact with individual microbial features depicted as the actual nucleotide sequence.

In [22]:
! qiime sample-classifier classify-samples \
  --i-table $data_dir/mags-table-filtered-milo-no-unassigned.qza \
  --m-metadata-file $data_dir/updog_metadata.tsv \
  --m-metadata-column subsistence_mode \
  --p-test-size 0.2 \
  --p-estimator RandomForestClassifier \
  --p-random-state 22 \
  --p-n-jobs 3 \
  --output-dir $data_dir/small-RF-classifier

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved SampleEstimator[Classifier] to: /home/jovyan/Group_Project/ML_Classifier/updog_data/small-RF-classifier/sample_estimator.qza
Saved FeatureData[Importance] to: /home/jovyan/Group_Project/ML_Classifier/updog_data/small-RF-classifier/feature_importance.qza
Saved SampleData[ClassifierPredictions] to: /home/jovyan/Group_Project/ML_Classifier/updog_data/small-RF-classifier/predictions.qza
Saved Visualization to: /home/jovyan/Group_Project/ML_Classifier/updog_data/small-RF-classifier/model_summary.qzv
Saved Visualization to: /home/jovyan/Group_Project/ML_Classifier/updog_data/small-RF-classifier/accuracy_results.qzv
Saved SampleData[Probabilities] to: /home/jovyan/

### 2.2 Evaluate trained classifier: Confusion matrix, accuracy & ROC

In [21]:
Visualization.load(f"{data_dir}/small-RF-classifier/accuracy_results.qzv")

<visualization: Visualization uuid: aab8009f-c068-4a3c-8e30-d0f50fa804ae>

### 2.2 Training another classifier

In [ ]:
! qiime sample-classifier classify-samples \
  --i-table $data_dir/mags-table-filtered-milo-no-unassigned.qza \
  --m-metadata-file $data_dir/updog_metadata.tsv \
  --m-metadata-column subsistence_mode \
  --p-test-size 0.2 \
  --p-estimator ExtraTreesClassifier \
  --p-random-state 22 \
  --p-n-jobs 3 \
  --output-dir $data_dir/small-ET-classifier

-> The ET classifier achieved more accurate results

In [26]:
Visualization.load(f"{data_dir}/small-ET-classifier/accuracy_results.qzv")

<visualization: Visualization uuid: a3852be0-976a-4644-aa45-59d686c08685>

### 2.3 Evaluate trained classifier: Individual predictions

In [27]:
! qiime metadata tabulate \
  --m-input-file $data_dir/small-RF-classifier/test_targets.qza \
  --m-input-file $data_dir/small-RF-classifier/predictions.qza \
  --m-input-file $data_dir/small-RF-classifier/probabilities.qza \
  --o-visualization $data_dir/small-RF-classifier/test_predprob.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/conda/lib/python3.10/site-packages/q2_sample_classifier/_transformer.py:76: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  lambda x: pd.to_numeric(x, errors='ignore')))
/opt/conda/lib/python3.10/site-packages/q2_sample_classifier/_transformer.py:76: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  lambda x: pd.to_numeric(x, errors='ignore')))
Saved Visualization to: /home/jovyan/Group_Project/ML_Classifier/updog_data/small-RF-classif

In [28]:
! qiime metadata tabulate \
  --m-input-file $data_dir/small-ET-classifier/test_targets.qza \
  --m-input-file $data_dir/small-ET-classifier/predictions.qza \
  --m-input-file $data_dir/small-ET-classifier/probabilities.qza \
  --o-visualization $data_dir/small-ET-classifier/test_predprob.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/conda/lib/python3.10/site-packages/q2_sample_classifier/_transformer.py:76: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  lambda x: pd.to_numeric(x, errors='ignore')))
/opt/conda/lib/python3.10/site-packages/q2_sample_classifier/_transformer.py:76: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  lambda x: pd.to_numeric(x, errors='ignore')))
Saved Visualization to: /home/jovyan/Group_Project/ML_Classifier/updog_data/small-ET-classif

### 2.4 Evaluate trained classifier: Feature importances 

Knowing that our classifier predicts `susbsistence_mode` quite accurately, we are now interested in knowing which microbial compositions are the most important ones for distinguishing the `subsistence_mode`. We can find a list of most predictive features in the produced output `feature_importance.qza` from our small-ET-classifier. 

In [29]:
! qiime metadata tabulate \
  --m-input-file $data_dir/small-ET-classifier/feature_importance.qza \
  --o-visualization $data_dir/small-ET-classifier/feature_importance.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: /home/jovyan/Group_Project/ML_Classifier/updog_data/small-ET-classifier/feature_importance.qzv


In [30]:
Visualization.load(f"{data_dir}/small-ET-classifier/feature_importance.qzv")

<visualization: Visualization uuid: e62fa60c-a196-4107-befd-876a3d2d35bb>

In [31]:
! qiime sample-classifier heatmap \
  --i-table $data_dir/mags-table-filtered-milo-no-unassigned.qza \
  --i-importance $data_dir/small-RF-classifier/feature_importance.qza \
  --m-sample-metadata-file $data_dir/updog_metadata.tsv  \
  --m-sample-metadata-column subsistence_mode \
  --p-group-samples \
  --p-feature-count 30 \
  --o-filtered-table $data_dir/small-ET-classifier/important-feature-table-top-30.qza \
  --o-heatmap $data_dir/small-ET-classifier/important-feature-heatmap.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: /home/jovyan/Group_Project/ML_Classifier/updog_data/small-ET-classifier/important-feature-heatmap.qzv
Saved FeatureTable[Frequency] to: /home/jovyan/Group_Project/ML_Classifier/updog_data/small-ET-classifier/important-feature-table-top-30.qza


In [ ]:
	•	Lighter colors → higher abundance
	•	Darker colors → lower abundance

In [33]:
Visualization.load(f"{data_dir}/small-ET-classifier/important-feature-heatmap.qzv")

<visualization: Visualization uuid: d90d2ae2-944a-4743-a38b-678aa276b6d4>